In [0]:
# ============================================================
# CELL 1: Read from Bronze Layer
# ============================================================

# Always read from the previous layer — never re-generate data.
# This ensures our pipeline is sequential and traceable.

df_bronze = spark.table("healthcare_db.bronze_patients")

print(f"✅ Loaded Bronze table: {df_bronze.count()} rows")
df_bronze.printSchema()

✅ Loaded Bronze table: 1000 rows
root
 |-- patient_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- admission_date: string (nullable = true)
 |-- discharge_date: string (nullable = true)
 |-- hospital: string (nullable = true)
 |-- treatment_cost: double (nullable = true)
 |-- readmitted: string (nullable = true)



In [0]:
# ============================================================
# CELL 2: Fix Data Types
# ============================================================

# Even though inferSchema=True was used, date columns often 
# come in as STRING. We need to cast them to proper DateType
# so we can do date arithmetic (like calculating days between dates)

from pyspark.sql.functions import col, to_date

df_typed = df_bronze \
    .withColumn("admission_date", to_date(col("admission_date"), "yyyy-MM-dd")) \
    .withColumn("discharge_date", to_date(col("discharge_date"), "yyyy-MM-dd")) \
    .withColumn("age",            col("age").cast("integer")) \
    .withColumn("treatment_cost", col("treatment_cost").cast("double"))

# Verify the new schema
df_typed.printSchema()
print("✅ Data types fixed!")

root
 |-- patient_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- hospital: string (nullable = true)
 |-- treatment_cost: double (nullable = true)
 |-- readmitted: string (nullable = true)

✅ Data types fixed!


In [0]:
# ============================================================
# CELL 3: Calculate Length of Stay
# ============================================================

# datediff(end, start) → returns number of days between two dates
# This is a KEY healthcare metric — longer stays = higher cost & complexity

from pyspark.sql.functions import datediff

df_los = df_typed.withColumn(
    "length_of_stay",
    datediff(col("discharge_date"), col("admission_date"))
)

# Quick sanity check — make sure no negative values exist
negative_stays = df_los.filter(col("length_of_stay") < 0).count()
print(f"❗ Records with negative length of stay: {negative_stays}")

display(df_los.select("patient_id", "admission_date", "discharge_date", "length_of_stay").limit(10))

❗ Records with negative length of stay: 0


patient_id,admission_date,discharge_date,length_of_stay
P0001,2023-10-17,2023-10-21,4
P0002,2023-11-24,2023-12-18,24
P0003,2022-08-27,2022-09-13,17
P0004,2023-08-27,2023-09-05,9
P0005,2022-12-11,2022-12-15,4
P0006,2022-02-14,2022-03-10,24
P0007,2023-10-06,2023-10-26,20
P0008,2022-10-24,2022-10-27,3
P0009,2022-06-16,2022-06-28,12
P0010,2023-07-01,2023-07-25,24


In [0]:
# ============================================================
# CELL 4: Standardize & Enrich Columns
# ============================================================

# Real-world data is messy — values like "male", "MALE", "M" 
# all mean the same thing. We standardize them.
# We also create an age_group column for easier analysis later.

from pyspark.sql.functions import upper, trim, when

df_clean = df_los \
    .withColumn("gender", trim(upper(col("gender")))) \
    .withColumn("diagnosis", trim(upper(col("diagnosis")))) \
    .withColumn("readmitted", trim(upper(col("readmitted")))) \
    .withColumn(
        "age_group",
        when(col("age") < 30, "18-29")
        .when(col("age") < 45, "30-44")
        .when(col("age") < 60, "45-59")
        .when(col("age") < 75, "60-74")
        .otherwise("75+")
    ) \
    .withColumn(
        "cost_category",
        when(col("treatment_cost") < 5000,  "Low")
        .when(col("treatment_cost") < 20000, "Medium")
        .otherwise("High")
    )

print("✅ Columns standardized and enriched!")
display(df_clean.limit(5))

✅ Columns standardized and enriched!


patient_id,name,age,gender,diagnosis,admission_date,discharge_date,hospital,treatment_cost,readmitted,length_of_stay,age_group,cost_category
P0001,Patient_1,21,OTHER,HEART FAILURE,2023-10-17,2023-10-21,Green Valley Medical,11548.93,YES,4,18-29,Medium
P0002,Patient_2,87,MALE,KIDNEY DISEASE,2023-11-24,2023-12-18,City General Hospital,1974.96,YES,24,75+,Low
P0003,Patient_3,21,OTHER,COVID-19,2022-08-27,2022-09-13,Lakeside Clinic,21266.23,NO,17,18-29,High
P0004,Patient_4,18,MALE,KIDNEY DISEASE,2023-08-27,2023-09-05,Sunrise Health Center,14254.63,YES,9,18-29,Medium
P0005,Patient_5,29,FEMALE,HYPERTENSION,2022-12-11,2022-12-15,Sunrise Health Center,42450.97,NO,4,18-29,High


In [0]:
# ============================================================
# CELL 5: Handle Nulls
# ============================================================

# Check null counts across all columns before saving
from pyspark.sql.functions import sum as spark_sum

null_counts = df_clean.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
])

print("🔍 Null counts in Silver layer:")
display(null_counts)

# Drop any rows where critical columns are null
df_silver = df_clean.dropna(subset=["patient_id", "admission_date", "discharge_date", "diagnosis"])

print(f"✅ Rows before null drop : {df_clean.count()}")
print(f"✅ Rows after null drop  : {df_silver.count()}")

🔍 Null counts in Silver layer:


patient_id,name,age,gender,diagnosis,admission_date,discharge_date,hospital,treatment_cost,readmitted,length_of_stay,age_group,cost_category
0,0,0,0,0,0,0,0,0,0,0,0,0


✅ Rows before null drop : 1000
✅ Rows after null drop  : 1000


In [0]:
# ============================================================
# CELL 6: Save as Silver Delta Table
# ============================================================

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_db.silver_patients")

print("✅ Silver Delta table saved!")

✅ Silver Delta table saved!


In [0]:
# ============================================================
# CELL 7: Verify Silver Table with SQL
# ============================================================

spark.sql("USE healthcare_db")

result = spark.sql("""
    SELECT 
        age_group,
        cost_category,
        ROUND(AVG(length_of_stay), 2)  AS avg_stay_days,
        ROUND(AVG(treatment_cost), 2)  AS avg_cost,
        COUNT(*)                        AS total_patients
    FROM healthcare_db.silver_patients
    GROUP BY age_group, cost_category
    ORDER BY age_group, cost_category
""")

display(result)

age_group,cost_category,avg_stay_days,avg_cost,total_patients
18-29,High,13.46,35158.83,90
18-29,Low,17.92,2515.69,13
18-29,Medium,15.05,10924.33,43
30-44,High,16.33,34046.88,136
30-44,Low,14.87,2585.27,15
30-44,Medium,17.72,12384.0,57
45-59,High,16.21,34654.22,131
45-59,Low,17.19,3196.4,16
45-59,Medium,16.11,11877.14,63
60-74,High,15.44,36002.43,120
